# NBME — Stage 3 & 4: Student Model Training + Inference

**Stage 3**: Student model training with pseudo labels + real data, 3-channel auxiliary targets, AWP  
**Stage 4**: CV evaluation, post-processing, ensemble with Optuna weight search

### Prerequisites (verify before running)
1. `train.csv` — real labeled data (columns: `id, case_num, pn_num, feature_num, annotation, location`)
2. `patient_notes.csv` — columns: `pn_num, case_num, pn_history`
3. `features.csv` — columns: `feature_num, case_num, feature_text`
4. `final_ensemble_pred_missing_rows.csv` — teacher pseudo labels (your teammate's output)
5. Pretrained teacher/base model weights in `./models/` directory

## Configuration

In [1]:
import gc, re, ast, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import json
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option('display.max_colwidth', 100)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.0.dev20260407+cu128
True
NVIDIA GeForce RTX 5090


### CFG configuration

In [2]:
class CFG:
    DATA_DIR = Path('.')
    PSEUDO_LABEL_PATH = Path('./final_ensemble_pred_missing_rows.csv')
    OUTPUT_DIR = Path('./outputs')
    
    model_name = './autodl-tmp/deberta-base'
    max_len = 512
    # As defined, three channels: Normal, Start, End
    num_labels = 3                   
    n_folds = 5
    train_folds = [0,1,2,3,4]
    epochs = 5
    # effective batch = batch_size * grad_accum
    batch_size = 12 
    grad_accum = 1                         
    lr = 2e-5
    weight_decay = 1e-2
    warmup_ratio = 0.0
    max_grad_norm = 1000
    scheduler = 'cosine'
    fp16 = True
    seed = 42

    # AWP
    use_awp = True
    awp_lr = 1.0
    awp_eps = 0.005
    awp_start_epoch = 3                 

    # filter pseudo labels below the threshold
    pseudo_confidence_threshold = 0.95
    
    # 10% real labels, 90% pseudo labels in each batch
    real_data_ratio = 0.7                      

    #  prob of dropping a sentence of real data
    aug_sentence_drop_p = 0.2                  

    # joint-probability threshold
    threshold = 0.4      
    # merge adjacent spans within N chars                
    merge_gap = 0                         

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 0

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {CFG.device}')
print(f'Model:  {CFG.model_name}')

Device: cuda
Model:  ./autodl-tmp/deberta-base


### Code reproductivity
fixes all random seeds to ensure fully reproducible results.

In [3]:
def seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed(CFG.seed)

## Data pre-processing

### Load real data

In [4]:

train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
pn_df = pd.read_csv(CFG.DATA_DIR / 'patient_notes.csv')
feature_df = pd.read_csv(CFG.DATA_DIR / 'features.csv')

# Transfrom data
# Parse location strings like "'0 3; 5 9'" -> list of (start, end) tuples
def parse_location(loc_str):
    """Convert location string to list of (start, end) int tuples."""
    # Return empty list if the location is missing or empty
    if pd.isna(loc_str) or loc_str in ('', '[]'):
        return []
    
    # Handle two formats:
    # Format 1: list-of-strings ["0 3", "5 9"] → use ast.literal_eval to parse
    # Format 2: raw string "0 3;5 9" → split by semicolon
    if loc_str.startswith('['):
        spans = ast.literal_eval(loc_str)
    else:
        spans = [s.strip() for s in loc_str.split(';')]
    result = []
    for span in spans:
        # Remove whitespace and quotes
        span = span.strip().strip("'")
        if span:
            # Split "0 3" into ["0", "3"] and convert to integers
            parts = span.split()
            if len(parts) == 2:
                result.append((int(parts[0]), int(parts[1])))
    # Example return element: [(0, 3), (5, 9)]
    return result

def fix_train_locations(df):
    fixes = {
        338: '["764 783"]',
        621: '["77 100"]',
        655: '["285 292;301 312", "285 287;296 312"]',
        1262: '["551 557;565 580"]',
        1265: '["131 135;181 212"]',
        1396: '["259 280"]',
        1591: '["176 184;201 212"]',
        1615: '["249 257;271 288"]',
        1664: '["822 824;907 924"]',
        1714: '["101 129"]',
        1929: '["531 539;549 561"]',
        2134: '["540 560;581 593"]',
        2191: '["32 57"]',
        2553: '["308 317;376 384"]',
        3124: '["549 557"]',
        3858: '["102 123", "102 112;125 141", "102 112;143 157", "102 112;159 171"]',
        4373: '["33 45"]',
        4763: '["5 16"]',
        4782: '["175 194"]',
        4908: '["700 723"]',
        6016: '["225 250"]',
        6192: '["197 218;236 260"]',
        6380: '["480 482;507 519", "480 482;499 503;512 519", "480 482;521 531", "480 482;533 545", "480 482;564 582"]',
        6562: '["290 320;327 337", "290 320;342 358"]',
        6862: '["288 296;324 363"]',
        7022: '["108 182"]',
        7422: '["102 121"]',
        8876: '["481 483;533 552"]',
        9027: '["92 102", "123 164"]',
        9938: '["89 117", "122 138", "368 402"]',
        9973: '["344 361"]',
        10513: '["600 611", "607 623"]',
        11551: '["386 400;443 461"]',
        11677: '["160 201"]',
        12124: '["325 337;349 366"]',
        12279: '["405 459;488 524"]',
        12289: '["353 400;488 524"]',
        13238: '["293 307", "321 331"]',
        13297: '["182 221", "182 213;225 234"]',
        13299: '["79 88", "409 418"]',
        13845: '["86 94;230 236", "86 94;237 256"]',
        14083: '["56 64;156 179"]',
    }
    df = df.copy()
    for idx, loc in fixes.items():
        if idx in df.index:
            df.loc[idx, 'location'] = loc
    return df

train_df = fix_train_locations(train_df)

train_df['location_parsed'] = train_df['location'].apply(parse_location)
train_df['is_real'] = True

feature_df.loc[27, 'feature_text'] = "Last-Pap-smear-1-year-ago"
# Merge patient note text and feature text into train_df
train_df = train_df.merge(pn_df[['pn_num', 'case_num', 'pn_history']], on=['pn_num', 'case_num'], how='left')
train_df = train_df.merge(feature_df[['feature_num', 'case_num', 'feature_text']], on=['feature_num', 'case_num'], how='left')


print(f'The number of train samples: {len(train_df)}')
train_df[['id', 'pn_num', 'feature_num', 'pn_history', 'feature_text', 'location_parsed', 'is_real']].head(3)

The number of train samples: 14300


,id,pn_num,feature_num,pn_history,feature_text,location_parsed,is_real
0,00016_000,16,0,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Family-history-of-MI-OR-Family-history-of-myocardial-infarction,"[(696, 724)]",True
1,00016_001,16,1,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Family-history-of-thyroid-disorder,"[(668, 693)]",True
2,00016_002,16,2,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Chest-pressure,"[(203, 217)]",True


### Pseudo Labels preprocessing

In [5]:
pseudo_df = pd.read_csv(CFG.PSEUDO_LABEL_PATH)
print(f'Pseudo label samples: {len(pseudo_df)}')
print(f'Confidence distribution:\n{pseudo_df["confidence"].describe().round(3)}')

Pseudo label samples: 612602
Confidence distribution:
count    612602.000
mean          0.682
std           0.420
min           0.000
25%           0.085
50%           0.957
75%           0.985
max           0.996
Name: confidence, dtype: float64


### Filter & Prepare Pseudo Labels
1. Keep only high-confidence pseudo labels (≥ 0.80)
2. merge patient note and feature text features so the format matches the real data.

In [6]:
# Keep rows with non-empty annotations AND confidence >= threshold
pseudo_filtered = pseudo_df[
    (pseudo_df['annotation'] != '[]') &
    (pseudo_df['confidence'] >= CFG.pseudo_confidence_threshold)
].copy()

print(f'After filtering (conf >= {CFG.pseudo_confidence_threshold}): {len(pseudo_filtered)} rows')

# Merge patient note and feature text
pseudo_filtered = pseudo_filtered.merge(
    pn_df[['pn_num', 'case_num', 'pn_history']], on=['pn_num', 'case_num'], how='left'
)
pseudo_filtered = pseudo_filtered.merge(
    feature_df[['feature_num', 'case_num', 'feature_text']], on=['feature_num', 'case_num'], how='left'
)

# Parse location string → list of (start, end) tuples
pseudo_filtered['location_parsed'] = pseudo_filtered['location'].apply(parse_location)

# Mark as pseudo (not real) so augmentation is skipped later
pseudo_filtered['is_real'] = False

# use pseudo id as string to match train_df for id column
pseudo_filtered = pseudo_filtered.rename(columns={'id': 'id'})

print(f'Pseudo label samples after merge: {len(pseudo_filtered)}')
pseudo_filtered[['id', 'pn_num', 'feature_num', 'confidence', 'location_parsed']].head(3)

After filtering (conf >= 0.95): 318629 rows
Pseudo label samples after merge: 318629


,id,pn_num,feature_num,confidence,location_parsed
0,pred_0,0,0,0.969671,"[(553, 566)]"
1,pred_1,0,1,0.957691,"[(576, 597)]"
2,pred_9,0,9,0.957886,"[(71, 85), (385, 418)]"


### Combine Real + Pseudo Data
Combine real and pseudo data.

In [7]:
# Align columns before concat
SHARED_COLS = ['id', 'case_num', 'pn_num', 'feature_num', 
               'pn_history', 'feature_text', 'location_parsed', 'is_real', 'confidence']

# real labels have confidence = 1.0
train_df['confidence'] = 1.0   

combined_df = pd.concat(
    [train_df[SHARED_COLS], pseudo_filtered[SHARED_COLS]], 
    ignore_index=True
)

print(f'Combined dataset (real + pseudo): {len(combined_df)} rows')
print(f'Real: {combined_df["is_real"].sum()}')
print(f'Pseudo: {(~combined_df["is_real"]).sum()}')

Combined dataset (real + pseudo): 332929 rows
Real: 14300
Pseudo: 318629


## Model Components

Core blocks of the student model pipeline: tokenisation, label construction, the PyTorch Dataset wrapper, the DeBERTa-based model architecture, the 3-channel BCE loss function, and the Adversarial Weight Perturbation (AWP) training technique.

### Tokenization & 3-Channel Label Construction
For pseudo labels we use confidence-weighted span-level soft labels rather than hard 0/1 labels.

We use a label vector `[normal, start, end]` for each token position:
- **Channel 0 (Normal)**: 1.0 (real data) or confidence score (pseudo label) if the token overlaps any annotated span
- **Channel 1 (Start)**: 1.0 (real data) or confidence score (pseudo label) if the token contains the first character of a span
- **Channel 2 (End)**: 1.0 (real data) or confidence score (pseudo label) if the token contains the last character of a span  
- **Channel 2 (End)**: 1.0 (real data) or confidence score (pseudo label)  if token is the *last* token of a span

In [8]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# Augmentation on real training data only
# Important: when text is edited, every char span after the removed sentence must be shifted.
def drop_random_sentence(text, location):
    """
    Randomly drop one sentence that does not overlap any annotated span.
    Returns (new_text, new_location). If no safe sentence exists, returns the original pair.
    """
    if not text or len(location) == 0:
        return text, location

    # Sentence spans exclude the whitespace separator after punctuation.
    spans = []
    start = 0
    for m in re.finditer(r'(?<=[.!?])\s+', text):
        spans.append((start, m.start(), m.end()))  # sentence_start, sentence_end, remove_until
        start = m.end()
    spans.append((start, len(text), len(text)))

    if len(spans) <= 1:
        return text, location

    candidates = []
    for s_start, s_end, remove_until in spans:
        overlaps = any(max(s_start, ann_s) < min(s_end, ann_e) for ann_s, ann_e in location)
        if not overlaps and s_start < s_end:
            candidates.append((s_start, remove_until))

    if not candidates:
        return text, location

    drop_start, drop_end = candidates[np.random.randint(len(candidates))]
    removed_len = drop_end - drop_start
    new_text = text[:drop_start] + text[drop_end:]

    new_location = []
    for s, e in location:
        if e <= drop_start:
            new_location.append((s, e))
        elif s >= drop_end:
            new_location.append((s - removed_len, e - removed_len))
        else:
            # Should not happen because candidates are non-overlapping, but keep labels sane.
            continue

    return new_text, new_location


def tokenize_and_build_labels(row, tokenizer, max_len=CFG.max_len, augment=False):
    """
    Tokenize [feature_text] + [pn_history] and build 3-channel per-token labels.
    Returns dict with input_ids, attention_mask, token_type_ids, labels, offset_mapping.
    """
    pn_text = str(row['pn_history'])
    feature_text = str(row['feature_text'])
    
    # list of (start, end) char tuples
    location = row['location_parsed']   
    is_real = row['is_real']
    confidence = float(row['confidence'])

    # Augmentation on real data
    if augment and is_real and np.random.random() < CFG.aug_sentence_drop_p:
        pn_text, location = drop_random_sentence(pn_text, location)

    # Tokenization
    # "fever" + "patient has fever and cough"
    # [CLS] fever [SEP] patient has fever and cough [SEP]
    encoding = tokenizer(
        feature_text,
        pn_text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_offsets_mapping=True,
        return_token_type_ids=True,
    )

    # list of (char_start, char_end) per token
    offset_mapping = encoding['offset_mapping']   
    
    # 0=feature tokens, 1=pn tokens, None=special 
    sequence_ids = encoding.sequence_ids()       

    # 3-channel float labels — shape (max_len, 3)
    labels = np.zeros((max_len, CFG.num_labels), dtype=np.float32)

    # Character level boolean arrays for the patient note text
    # 1 if it is in the span, 0 otherwise
    pn_len = len(pn_text)
    char_normal = np.zeros(pn_len, dtype=np.float32)
    char_start = np.zeros(pn_len, dtype=np.float32)
    char_end = np.zeros(pn_len, dtype=np.float32)
    
    # Soft labels for pseudo labels, hard 1.0 for real labels
    label_value = confidence if not is_real else 1.0

    for (s, e) in location:
        s, e = int(s), int(e)
        if s >= pn_len or e > pn_len:
            continue
        char_normal[s:e]  = label_value
        if s < pn_len:
            # first char of span
            char_start[s] = label_value    
        if e - 1 >= 0 and e - 1 < pn_len:
            # last char of span
            char_end[e - 1] = label_value  

    # Map char-level labels to token-level labels
    # Only process pn tokens (seq_id=1); skip feature tokens and special tokens
    for idx, (seq_id, (tok_s, tok_e)) in enumerate(zip(sequence_ids, offset_mapping)):
        if seq_id != 1:
            continue
        if tok_s == tok_e:
            continue
        # If any character in this token's range is labelled, mark the token
        labels[idx, 0] = char_normal[tok_s:tok_e].max()   
        labels[idx, 1] = char_start[tok_s:tok_e].max()    
        labels[idx, 2] = char_end[tok_s:tok_e].max()  
    # for idx, (seq_id, (tok_s, tok_e)) in enumerate(zip(sequence_ids, offset_mapping)):
    #     if seq_id != 1:
    #         continue
    #     if tok_s == tok_e:
    #         continue
    #     # Channel 0: normal (always present)
    #     labels[idx, 0] = char_normal[tok_s:tok_e].max()
    #     # Channel 1: start (only if num_labels > 1)
    #     if CFG.num_labels > 1:
    #         labels[idx, 1] = char_start[tok_s:tok_e].max()
    #     # Channel 2: end (only if num_labels > 2)
    #     if CFG.num_labels > 2:
    #         labels[idx, 2] = char_end[tok_s:tok_e].max()

    return {
        'input_ids': np.array(encoding['input_ids'], dtype=np.int64),
        'attention_mask': np.array(encoding['attention_mask'], dtype=np.int64),
        'token_type_ids': np.array(encoding['token_type_ids'], dtype=np.int64),
        'labels': labels,          # (max_len, 3)
        'offset_mapping': np.array(offset_mapping, dtype=np.int32),
        'sequence_ids': np.array([s if s is not None else -1 for s in sequence_ids], dtype=np.int8),
    }


### Dataset
Wraps the dataframe rows into a format the DataLoader can iterate over.

In [9]:
class NBMEDataset(Dataset):
    """
    Handles both real and pseudo-labeled rows.
    For training: returns labels (3-channel).
    """
    def __init__(self, df, tokenizer, is_train=True, augment=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_train = is_train
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = tokenize_and_build_labels(row, self.tokenizer, augment=self.augment)

        item = {
            'input_ids': torch.tensor(encoded['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoded['attention_mask'], dtype=torch.long),
            'token_type_ids': torch.tensor(encoded['token_type_ids'], dtype=torch.long),
            'offset_mapping': torch.tensor(encoded['offset_mapping'], dtype=torch.long),
            'sequence_ids': torch.tensor(encoded['sequence_ids'],   dtype=torch.long),
        }
        if self.is_train:
            item['labels'] = torch.tensor(encoded['labels'], dtype=torch.float32)
        return item

### 3 channels token classifier
Backbone model with a 3-output linear head. Each token produces three logits: Normal, Start, End.

In [10]:
class NBMEModel(nn.Module):
    """
    DeBERTa-based token classifier with 3 output channels:
    Channel 0: Normal (is this token part of the clinical feature?)
    Channel 1: Start (is this token the START of the feature span?)
    Channel 2: End (is this token the END of the feature span?)
    """
    def __init__(self, model_name, num_labels=CFG.num_labels):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob       = 0.0
        config.attention_probs_dropout_prob = 0.0

        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.dropout  = nn.Dropout(0.2)
        self.head = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # (B, L, hidden)
        seq_out = out.last_hidden_state          
        seq_out = self.dropout(seq_out)
        # (B, L, 3)
        logits  = self.head(seq_out)             
        return logits                         

### Loss Function
Binary cross-entropy applied. Padding and special tokens are masked out. Supports soft labels for pseudo data.

In [11]:
class ThreeChannelBCELoss(nn.Module):
    """
    Binary cross-entropy loss over all 3 channels simultaneously.
    Handles soft labels naturally via BCEWithLogitsLoss.
    Masks out special tokens and padding (sequence_id != 1).
    """
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, labels, sequence_ids):
        loss = self.bce(logits, labels)  # (B, L, 3)
        mask = (sequence_ids == 1).unsqueeze(-1).float()  # (B, L, 1)
        loss = (loss * mask).sum() / (mask.sum() * logits.size(-1)).clamp(min=1.0)
        return loss

criterion = ThreeChannelBCELoss()

### AWP (Adversarial Weight Perturbation)
Perturbs model weights in the direction of maximum loss to force the model to improve generalization performance.

In [12]:
class AWP:
    """
    Adversarial Weight Perturbation.
    1. Normal forward+backward → compute clean gradients
    2. Perturb weights by adding scaled gradient noise (makes the model "worst case")
    3. Forward+backward again on perturbed weights → get adversarial gradients
    4. Restore original weights; optimizer.step() uses adversarial gradients
    """
    def __init__(self, model, optimizer, adv_lr=1.0, adv_eps=0.01):
        self.model = model
        self.optimizer = optimizer
        # step size for computing perturbation
        self.adv_lr = adv_lr
        # max allowed perturbation magnitude
        self.adv_eps = adv_eps
        # stores original weights
        self.backup = {}          

    def attack(self):
        """Compute and apply weight perturbation"""
        # small epsilon to avoid division by zero
        e1 = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                # restore original weight later
                self.backup[name] = param.data.clone()
                # Compute gradient norm 
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    # aompute perturbation direction: scaled gradient unit vector
                    r_adv = self.adv_lr * param.grad / (norm + e1)
                    # apply perturbation to weights
                    param.data.add_(r_adv)
                    # clamp perturbation to stay within epsilon range
                    param.data = self._clamp(param.data, self.backup[name], self.adv_eps)

    def restore(self):
        """Restore original weights after adversarial step."""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                # Reset weight back to pre-attack value
                param.data = self.backup[name]
        # clear backup
        self.backup = {}

    @staticmethod
    def _clamp(data, original, eps):
        # ensure perturbed weight stays within [original-eps, original+eps]
        return torch.clamp(data, original - eps, original + eps)
    

### Training and Validation functions

In [13]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, awp, epoch):
    model.train()
    total_loss = 0.0
    use_awp_this_epoch = CFG.use_awp and (epoch >= CFG.awp_start_epoch)

    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc=f'Epoch {epoch} train')):
        # token ids
        input_ids      = batch['input_ids'].to(CFG.device)
        # 1=real token, 0=padding
        attention_mask = batch['attention_mask'].to(CFG.device)
        # 0=feature, 1=patient note
        token_type_ids = batch['token_type_ids'].to(CFG.device)
        # (B, L, 3) ground truth
        labels         = batch['labels'].to(CFG.device)
        # used to mask loss
        sequence_ids   = batch['sequence_ids'].to(CFG.device)

        is_accum_step = ((step + 1) % CFG.grad_accum == 0)

        # normal forward pass
        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels, sequence_ids)
            loss   = loss / CFG.grad_accum

        scaler.scale(loss).backward()

        # only update model every grad_accum steps
        if is_accum_step:
            scaler.unscale_(optimizer)

            if use_awp_this_epoch:
                awp.attack()
                try:
                    with torch.amp.autocast('cuda', enabled=CFG.fp16):
                        adv_logits = model(input_ids, attention_mask, token_type_ids)
                        adv_loss = criterion(adv_logits, labels, sequence_ids) / CFG.grad_accum
                    adv_loss.backward()
                finally:
                    awp.restore()

            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            if (not CFG.fp16) or (scaler.get_scale() >= old_scale):
                scheduler.step()

        # accumulate loss
        total_loss += loss.item() * CFG.grad_accum

    # average loss
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    """
    Return:
      all_preds: list of (B, L, 3) sigmoid arrays
      all_labels: list of (B, L, 3) ground truth arrays
      all_offset_mappings: list of (B, L, 2) arrays
      all_sequence_ids: list of (B, L) arrays
    """
    model.eval()
    all_preds, all_labels, all_offsets, all_seqids = [], [], [], []

    for batch in tqdm(loader, desc='Validation'):
        input_ids = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        token_type_ids = batch['token_type_ids'].to(CFG.device)

        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)

        # (B, L, 3)
        preds = torch.sigmoid(logits).cpu().numpy()          
        all_preds.append(preds)
        # (B, L, 3)
        all_labels.append(batch['labels'].numpy())      
         # (B, L, 2)     
        all_offsets.append(batch['offset_mapping'].numpy()) 
         # (B, L)
        all_seqids.append(batch['sequence_ids'].numpy())    

    return (
        np.concatenate(all_preds,   axis=0),
        np.concatenate(all_labels,  axis=0),
        np.concatenate(all_offsets, axis=0),
        np.concatenate(all_seqids,  axis=0),
    )

## Post Processing: Convert probabilities to Character Spans
Each token's normal-channel probability is mapped back to character level.
Characters exceeding the threshold are grouped into spans.

In [14]:
def preds_to_char_spans(probs_3ch, offset_mapping, sequence_ids,
                         threshold=CFG.threshold, merge_gap=CFG.merge_gap, text=None):
    if text is not None:
        char_probs = np.zeros(len(text), dtype=np.float32)
    else:
        max_end = int(offset_mapping[:, 1].max()) if len(offset_mapping) else 0
        char_probs = np.zeros(max_end, dtype=np.float32)

    prob_normal = probs_3ch[:, 0]
    for prob, (tok_s, tok_e), seq_id in zip(prob_normal, offset_mapping, sequence_ids):
        if seq_id != 1:
            continue
        tok_s, tok_e = int(tok_s), int(tok_e)
        if tok_s >= tok_e or tok_s >= len(char_probs):
            continue
        tok_e = min(tok_e, len(char_probs))
        char_probs[tok_s:tok_e] = np.maximum(char_probs[tok_s:tok_e], float(prob))

    active = np.where(char_probs >= threshold)[0]
    if len(active) == 0:
        return ''

    spans = []
    start = prev = int(active[0])
    for idx in active[1:]:
        idx = int(idx)
        if idx <= prev + 1 + merge_gap:
            prev = idx
        else:
            spans.append((start, prev + 1))
            start = prev = idx
    spans.append((start, prev + 1))

    return ';'.join(f'{s} {e}' for s, e in spans)



def strip_whitespace_from_spans(span_str, pn_text):
    """
    Post-processing: trim leading/trailing whitespace within each span
    by adjusting character indices to exclude whitespace chars.
    
    This mirrors an important post-processing step.
    """
    if not span_str:
        return ''

    result = []
    for part in span_str.split(';'):
        part = part.strip()
        if not part:
            continue
        s, e = int(part.split()[0]), int(part.split()[1])
        # move start index to a non-whitespace char
        while s < e and pn_text[s].isspace():
            s += 1
        # move end index to a non-whitespace char
        while e > s and pn_text[e - 1].isspace():
            e -= 1
        # keep span if it still has content after trimming
        if s < e:
            result.append(f'{s} {e}')

    # rejoin all cleaned spans with semicolons
    return ';'.join(result)

## Threshold search

In [15]:
def parse_span_str(span_str):
    """Parse '0 3;5 9' → set of integer char indices {0,1,2,5,6,7,8}"""
    indices = set()
    if not span_str or pd.isna(span_str):
        return indices
    for part in str(span_str).split(';'):
        part = part.strip()
        if part:
            s, e = int(part.split()[0]), int(part.split()[1])
            indices.update(range(s, e))
    return indices


def compute_micro_f1(pred_spans, true_spans):
    """
    Compute micro-averaged F1 across all samples.
    Args:
        pred_spans: list of span strings (predictions)
        true_spans: list of span strings (ground truth)
    """
    total_tp = total_fp = total_fn = 0

    for pred, true in zip(pred_spans, true_spans):
        pred_set = parse_span_str(pred)
        true_set = parse_span_str(true)

        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {'tp': total_tp, 'fp': total_fp, 'fn': total_fn,
            'precision': precision, 'recall': recall, 'f1': f1}

def find_best_threshold_for_val(preds, offsets, seq_ids, val_df, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.71, 0.02)

    true_spans = [
        ';'.join(f'{s} {e}' for s, e in row.location_parsed)
        for row in val_df.itertuples()
    ]

    best = {'threshold': CFG.threshold, 'f1': -1.0, 'precision': 0.0, 'recall': 0.0}

    for thr in thresholds:
        pred_spans = []
        for i, row in enumerate(val_df.itertuples()):
            span_str = preds_to_char_spans(
                preds[i], offsets[i], seq_ids[i], threshold=float(thr), text=str(row.pn_history)
            )
            span_str = strip_whitespace_from_spans(span_str, str(row.pn_history))
            pred_spans.append(span_str)

        metrics = compute_micro_f1(pred_spans, true_spans)
        if metrics['f1'] > best['f1']:
            best = {'threshold': float(thr), **metrics}

    return best

## CV Split (StratifiedGroupKFold)
Grouping by `pn_num` prevents data leakage. 
Stratifying by `case_num` ensures balanced case distribution across folds.

In [16]:
def create_cv_folds(df, n_folds=CFG.n_folds):
    df = df.copy()
    df['fold'] = -1
    
    sgkf = StratifiedGroupKFold(n_splits=n_folds)
    for fold_idx, (_, val_idx) in enumerate(
        sgkf.split(df, y=df['case_num'], groups=df['pn_num'])
    ):
        df.loc[df.index[val_idx], 'fold'] = fold_idx
    
    print('Fold distribution:')
    print(df.groupby('fold')['is_real'].agg(['sum', 'count']).rename(
        columns={'sum': 'real', 'count': 'total'}
    ))
    return df

# only assign folds to real data, pseudo labels are used for training only
real_df = combined_df[combined_df['is_real']].copy()
real_df = create_cv_folds(real_df)

# put pseudo labels into its dataframe
pseudo_only_df = combined_df[~combined_df['is_real']].copy()
print(f'\nReal: {len(real_df)} | Pseudo: {len(pseudo_only_df)}')

Fold distribution:
      real  total
fold             
0     2860   2860
1     2860   2860
2     2860   2860
3     2860   2860
4     2860   2860

Real: 14300 | Pseudo: 318629


### Main Training Loop

In [17]:
def get_optimizer_and_scheduler(model, num_training_steps):
    """AdamW with differential learning rates."""
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    param_groups = [
        # backbone: lower lr, weight decay
        {'params': [p for n, p in model.backbone.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'lr': CFG.lr, 'weight_decay': CFG.weight_decay},
        # backbone parameters without weight decay
        {'params': [p for n, p in model.backbone.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'lr': CFG.lr, 'weight_decay': 0.0},
        # classification head uses the same lr as the backbone
        {'params': model.head.parameters(),
         'lr': CFG.lr , 'weight_decay': 0.0},
    ]

    optimizer = torch.optim.AdamW(param_groups)

    # warmup: gradually increase lr from 0 to CFG.lr
    num_warmup = int(num_training_steps * CFG.warmup_ratio)
    # lr decreases linearly after warmup
    if CFG.scheduler == 'cosine':
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=num_warmup,
            num_training_steps=num_training_steps
        )
    else:
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=num_warmup,
            num_training_steps=num_training_steps
        )
    return optimizer, scheduler


def train_fold(fold, real_df, pseudo_df):
    print(f' FOLD {fold}')

    # split real data into current fold is validation and rest is training
    val_real = real_df[real_df['fold'] == fold]
    train_real = real_df[real_df['fold'] != fold]

    # Combine real training data with pseudo labels while enforcing the intended ratio.
    # # pseudo labels enabled, sample pseudo_train to match the target real/pseudo ratio.
    if len(pseudo_df) > 0 and CFG.real_data_ratio > 0:
        desired_pseudo = int(len(train_real) * (1 - CFG.real_data_ratio) / CFG.real_data_ratio)
        n_pseudo = min(len(pseudo_df), desired_pseudo)
        pseudo_train = pseudo_df.sample(n=n_pseudo, random_state=CFG.seed + fold).reset_index(drop=True)
    else:
        pseudo_train = pseudo_df.iloc[:0].copy()

    # combine real training data with pseudo labels and shuffle
    # train_combined = pd.concat([train_real, pseudo_train], ignore_index=True)
    train_combined = train_real.copy()
    train_combined = train_combined.sample(frac=1, random_state=CFG.seed).reset_index(drop=True)

    print(f'Train: {len(train_combined)} (real: {len(train_real)}, pseudo: {len(pseudo_train)})')
    print(f'Val: {len(val_real)} (real only)')
    
    train_ds = NBMEDataset(train_combined, tokenizer, is_train=True, augment=True)
    val_ds = NBMEDataset(val_real, tokenizer, is_train=True, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=True
    )
    
    # use 2x batch size for validation since no gradients needed
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True
    )

    # initialize model, optimizer, scheduler, scaler and AWP
    model = NBMEModel(CFG.model_name, num_labels=CFG.num_labels).to(CFG.device)
    num_steps = len(train_loader) * CFG.epochs // CFG.grad_accum
    optimizer, scheduler = get_optimizer_and_scheduler(model, num_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=CFG.fp16)
    awp    = AWP(model, optimizer, adv_lr=CFG.awp_lr, adv_eps=CFG.awp_eps)

    best_f1    = 0.0
    best_path  = CFG.OUTPUT_DIR / f'model_fold{fold}.pt'

    for epoch in range(CFG.epochs):
        # get average loss
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, awp, epoch)

        # run validation and collect probabilities
        preds, labels, offsets, seq_ids = validate(model, val_loader)

        # convert probabilities to span strings and compare with ground truth
        metrics = find_best_threshold_for_val(preds, offsets, seq_ids, val_real)
        f1 = metrics['f1']

        # tag epoch log with AWP indicator if active
        awp_tag = '(+AWP)' if CFG.use_awp and epoch >= CFG.awp_start_epoch else ''
        print(f'Epoch {epoch}: loss={train_loss:.4f}  F1={f1:.4f}  '
              f'thr={metrics["threshold"]:.2f}  '
              f'P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}  {awp_tag}')

        # Save checkpoint for the best F1
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)
            print(f'  ✓ Saved best model (F1={best_f1:.4f})')

    # free GPU memory before next fold
    del model, optimizer, scheduler, scaler, awp, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

    print(f'Fold {fold} best CV F1: {best_f1:.4f}')
    return best_f1

### Run training across selected folds

In [18]:
# train one model per fold and collect validation F1 scores
fold_scores = {}

print(f"train fold is: {CFG.train_folds}")
for fold in CFG.train_folds:
    # train all folds except current, validate on current fold
    score = train_fold(fold, real_df, pseudo_only_df)
    # store best F1 score
    fold_scores[fold] = score

print(f'fold scores: {fold_scores}')
print('CROSS-VALIDATION RESULTS')
for fold, score in fold_scores.items():
    print(f'Fold {fold}: F1 = {score:.4f}')
    
# key metric: Mean CV F1
mean_cv = np.mean(list(fold_scores.values()))
print(f'\n  Mean CV F1: {mean_cv:.4f}')

train fold is: [0, 1, 2, 3, 4]
 FOLD 0
Train: 11440 (real: 11440, pseudo: 4902)
Val: 2860 (real only)


Epoch 0 train:   0%|          | 0/954 [00:00<?, ?it/s]

Validation:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0: loss=0.0169  F1=0.8074  thr=0.12  P=0.7601  R=0.8610  
  ✓ Saved best model (F1=0.8074)


Epoch 1 train:   0%|          | 0/954 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 3: loss=0.0034  F1=0.8116  thr=0.28  P=0.8095  R=0.8136  (+AWP)


Epoch 4 train:   0%|          | 0/954 [00:00<?, ?it/s]

Validation:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 4: loss=0.0027  F1=0.8254  thr=0.26  P=0.8011  R=0.8513  (+AWP)
Fold 4 best CV F1: 0.8345
fold scores: {0: 0.8443564837524091, 1: 0.8212329677073196, 2: 0.8437313355559352, 3: 0.8309294866795379, 4: 0.8345176671729523}
CROSS-VALIDATION RESULTS
Fold 0: F1 = 0.8444
Fold 1: F1 = 0.8212
Fold 2: F1 = 0.8437
Fold 3: F1 = 0.8309
Fold 4: F1 = 0.8345

  Mean CV F1: 0.8350


## Inference

Loads a saved model checkpoint and runs forward pass on a given dataframe, returning per-token sigmoid probabilities of shape (N, max_len, 3). Used to collect OOF predictions from each fold model for validation and OOF evaluation.

In [19]:
# disable gradient computation to save memory during inference
@torch.no_grad()
def run_inference(model_path, test_df):
    """
    Run inference with a saved model checkpoint.
    Returns np.array of shape (N, max_len, 3) probabilities.
    """
    
    # load model and restore saved weights
    model = NBMEModel(CFG.model_name, num_labels=CFG.num_labels)
    model.load_state_dict(torch.load(model_path, map_location=CFG.device))
    model.to(CFG.device)
    model.eval()

    # Add dummy columns so tokenize_and_build_labels doesn't crash
    # (inference data has no labels, so we fill with empty/default values)
    inf_df = test_df.copy()
    if 'location_parsed' not in inf_df.columns:
        inf_df['location_parsed'] = [[] for _ in range(len(inf_df))]
    if 'is_real' not in inf_df.columns:
        # mark as non-real so augmentation is never applied during inference
        inf_df['is_real'] = False
    if 'confidence' not in inf_df.columns:
        inf_df['confidence'] = 0.0

    # dataset and dataloader
    ds = NBMEDataset(inf_df, tokenizer, is_train=False)
    loader = DataLoader(ds, batch_size=CFG.batch_size * 2, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)

    all_probs = []
    all_offsets = []
    all_seq_ids = []

    for batch in tqdm(loader, desc=f'Inference: {Path(model_path).name}'):
        input_ids      = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        token_type_ids = batch['token_type_ids'].to(CFG.device)

        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)

        # apply sigmoid to convert logits to probabilities within the range [0, 1]
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_offsets.append(batch['offset_mapping'].numpy())
        all_seq_ids.append(batch['sequence_ids'].numpy())

    # free GPU memory
    del model
    torch.cuda.empty_cache()

    # merge all batches into single arrays of shape (N, max_len, 3)
    return (
        np.concatenate(all_probs,   axis=0),
        np.concatenate(all_offsets, axis=0),
        np.concatenate(all_seq_ids, axis=0),
    )

## OOF Evaluation & Threshold Search

Collect Out-Of-Fold predictions from each trained fold model, then search for the optimal decision threshold that maximises OOF F1 score.

In [ ]:
# Initialize arrays to store OOF predictions for all real data rows
oof_preds = np.zeros((len(real_df), CFG.max_len, CFG.num_labels), dtype=np.float32)
oof_offsets = np.zeros((len(real_df), CFG.max_len, 2), dtype=np.int32)
oof_seq_ids = np.zeros((len(real_df), CFG.max_len), dtype=np.int8)

for fold in CFG.train_folds:
    model_path = CFG.OUTPUT_DIR / f'model_fold{fold}.pt'
    if not model_path.exists():
        print(f'Skipping fold {fold} — no checkpoint found')
        continue

    val_real_fold = real_df[real_df['fold'] == fold]
    probs, offsets_f, seq_ids_f = run_inference(model_path, val_real_fold)

    val_indices = real_df.index[real_df['fold'] == fold]
    pos = real_df.index.get_indexer(val_indices)

    oof_preds[pos] = probs
    oof_offsets[pos] = offsets_f
    oof_seq_ids[pos] = seq_ids_f

print('OOF probs max:', oof_preds.max())
print('OOF probs mean:', oof_preds.mean())


def oof_f1_score(probs, threshold):
    pred_spans, true_spans = [], []

    for i, row in enumerate(real_df.itertuples()):
        span_str = preds_to_char_spans(
            probs[i],
            oof_offsets[i],
            oof_seq_ids[i],
            threshold=threshold,
            text=str(row.pn_history),
        )
        span_str = strip_whitespace_from_spans(span_str, str(row.pn_history))
        pred_spans.append(span_str)

        true_spans.append(';'.join(f'{s} {e}' for s, e in row.location_parsed))

    metrics = compute_micro_f1(pred_spans, true_spans)
    return metrics


best_threshold = None
best_oof_metrics = None
best_oof_f1 = -1

for threshold in np.arange(0.10, 0.91, 0.02):
    metrics = oof_f1_score(oof_preds, float(threshold))
    if metrics['f1'] > best_oof_f1:
        best_oof_f1 = metrics['f1']
        best_threshold = float(threshold)
        best_oof_metrics = metrics

print(f'Best OOF F1:        {best_oof_metrics["f1"]:.4f}')
print(f'Best OOF Precision: {best_oof_metrics["precision"]:.4f}')
print(f'Best OOF Recall:    {best_oof_metrics["recall"]:.4f}')
print(f'Best threshold:     {best_threshold:.3f}')


Inference: model_fold0.pt:   0%|          | 0/120 [00:00<?, ?it/s]

Inference: model_fold1.pt:   0%|          | 0/120 [00:00<?, ?it/s]

Inference: model_fold2.pt:   0%|          | 0/120 [00:00<?, ?it/s]

## Save Experiment Record
Save the current real-only 5-fold baseline configuration and CV results to JSON for reproducibility and later ablation comparison.


In [ ]:
experiment = {
    'name': "bl_real_3_channel_5_fold_5epochs_0.95_ratio_0.7_awp_warm_0_max_grad_1000_aug",
    'config': {
        'model_name': CFG.model_name,
        'train_folds': CFG.train_folds,
        'epochs': CFG.epochs,
        'batch_size': CFG.batch_size,
        'grad_accum': CFG.grad_accum,
        'lr': CFG.lr,
        'use_awp': CFG.use_awp,
        'threshold': CFG.threshold,
        'pseudo_confidence_threshold': CFG.pseudo_confidence_threshold,
        'use_pseudo': False,
        'real_data_ratio': CFG.real_data_ratio,
        'n_folds': CFG.n_folds,
        'merge_gap': CFG.merge_gap,
    },
    'cv_results': fold_scores,
    'mean_cv_f1': float(np.mean(list(fold_scores.values()))),
    'oof_results': {
        'f1': float(best_oof_metrics['f1']),
        'precision': float(best_oof_metrics['precision']),
        'recall': float(best_oof_metrics['recall']),
        'best_threshold': float(best_threshold),
    }
}

save_path = CFG.OUTPUT_DIR / 'bl_real_3_channel_5_fold_5epochs_0.95_ratio_0.7_awp_warm_0_max_grad_1000_aug.json'

with open(save_path, 'w') as f:
    json.dump(experiment, f, indent=2)

print(f'Results saved to {save_path}')
print(json.dumps(experiment, indent=2))
